# PCIbex experiment results

Run the cells in order. `event_table` contains one row per **logged** experimental keystroke, with trial metadata and choices alongside each event. Practice and instructions are excluded. Session numbers distinguish separate submissions, even if a participant returns.

**Timing:** all times are milliseconds and are calculated in chronological order before sorting the table. `since_previous_key_ms` is the elapsed time since the preceding logged experimental key in the same submission, including across trials. `within_trial_ms` resets at each trial. These columns include context and question events. The context key is the first logged key in its trial, so its within-trial interval is missing; the cross-trial interval is not a context reading time. Question intervals can include transitions or fixed delays. Region reading times are the interval from the preceding region key to that region's key: for example, `r1_ms` runs from `r0` to `r1`. `r0` starts the sentence, not a word. The interval before the choice includes the 500 ms fixation and is **not** a choice-onset RT. As in the script, `cb` equal to `y` swaps the sides; every other value, including a blank, uses A on the left.

The preview space key has no `.log()` in the script. It, other unlogged keys, exact display-onset times and consequently true preview/choice-onset RTs cannot be reconstructed. This notebook does not infer or fabricate them. The file contains participant identifiers; keep any exports private.

In [1]:
import csv
import re
from pathlib import Path
import pandas as pd
from IPython.display import display

# Works when the notebook is run from either new/ or the repository root.
RESULTS = Path('results_01.csv')
if not RESULTS.exists():
    RESULTS = Path('new') / 'results_01.csv'

def read_pcibex(path):
    """Read the numbered, changing column descriptions in a PCIbex export."""
    records = []
    columns = {}
    session = 0
    with path.open(encoding='utf-8-sig', newline='') as source:
        for line_number, line in enumerate(source, 1):
            if line.startswith('# Results on '):
                session += 1
                columns = {}
            match = re.match(r'^#\s*(\d+)\.\s*(.*?)\.\s*$', line)
            if match:
                columns[int(match.group(1)) - 1] = match.group(2)
            elif line.startswith('#') or not line.strip():
                continue
            else:
                if not session:
                    raise ValueError(f'No results block before line {line_number}')
                values = next(csv.reader([line]))
                names = [columns.get(i) for i in range(len(values))]
                if any(name is None for name in names):
                    raise ValueError(f'Missing column description at line {line_number}')
                # participant_id appears twice: typed ID first, URL parameter second.
                seen = {}
                unique_names = []
                for name in names:
                    seen[name] = seen.get(name, 0) + 1
                    unique_names.append(name if seen[name] == 1 else f'{name}_{seen[name]}')
                record = dict(zip(unique_names, values))
                record.update(session=session, source_line=line_number)
                records.append(record)
    if not records:
        raise ValueError(f'No result rows in {path}')
    return pd.DataFrame(records)

raw = read_pcibex(RESULTS)
print(f'Read {len(raw):,} rows in {raw.session.nunique()} submission blocks from {RESULTS}')

Read 5,800 rows in 10 submission blocks from results_01.csv


In [2]:
trial_keys = ['session', 'Order number of item', 'Inner element number']
metadata = ['participant_id', 'prolific_pid', 'no', 'item', 'exp', 'cnd',
            'counterbalancing', 'a', 'b', 'correct', 'vpx', 'def', 'pst', 'mlt', 'alt']
experiment = raw.loc[raw['Label'].eq('experiment')].copy()
if experiment.empty:
    raise ValueError('No experiment rows found')
if 'counterbalancing' not in experiment:
    raise ValueError('Counterbalancing column is missing from the results')
experiment['EventTime'] = pd.to_numeric(experiment['EventTime'], errors='raise')
for column in metadata:
    if column not in experiment:
        experiment[column] = pd.NA
    experiment[column] = experiment[column].replace({'NULL': pd.NA, '': pd.NA})

# Order number identifies the design item, not its randomized presentation order.
starts = experiment.loc[experiment['Parameter'].eq('_Trial_') & experiment['Value'].eq('Start')].copy()
if starts.duplicated(trial_keys).any():
    raise ValueError('Repeated trial identifiers in a submission; inspect the raw results')
starts = starts.sort_values(['session', 'EventTime', 'source_line'])
starts['trial_number'] = starts.groupby('session').cumcount() + 1
trials = starts[trial_keys + ['trial_number', 'EventTime'] + metadata].rename(
    columns={'EventTime': 'trial_start_ms', 'counterbalancing': 'cb', 'a': 'option_a', 'b': 'option_b'})
if trials.empty:
    raise ValueError('No experimental trial starts found')
for column in ['no', 'item']:
    trials[column] = pd.to_numeric(trials[column], errors='raise').astype('Int64')
for column in ['exp', 'pst', 'mlt']:
    trials[column] = pd.to_numeric(trials[column], errors='raise').astype('Int64')

events = experiment.loc[experiment['PennElementType'].eq('Key') & experiment['Parameter'].eq('PressedKey'),
                        trial_keys + ['source_line', 'EventTime', 'PennElementName', 'Value']].copy()
events = events.rename(columns={'EventTime': 'timestamp_ms', 'PennElementName': 'stage', 'Value': 'key'})
events['key'] = events['key'].map(lambda x: 'SPACE' if x == ' ' else x.upper())
events = events.merge(trials, on=trial_keys, how='left', validate='many_to_one', indicator=True)
if not events['_merge'].eq('both').all():
    raise ValueError('Some key events have no matching experimental trial start')
events = events.drop(columns='_merge').sort_values(['session', 'timestamp_ms', 'source_line']).reset_index(drop=True)
events['key_number_in_trial'] = events.groupby(trial_keys, sort=False).cumcount() + 1
events['since_previous_key_ms'] = events.groupby('session')['timestamp_ms'].diff()
events['within_trial_ms'] = events.groupby(trial_keys, sort=False)['timestamp_ms'].diff()
events['previous_stage'] = events.groupby(trial_keys, sort=False)['stage'].shift()
if (events['timestamp_ms'] < events['trial_start_ms']).any():
    raise ValueError('A key event precedes its trial start')

# Interpret the logged choice exactly as in main.js: y swaps left and right.
choices = events.loc[events.stage.eq('choice_key')].copy()
if choices.duplicated(trial_keys).any():
    raise ValueError('More than one logged choice in a trial; inspect events')
if not choices['key'].isin(['F', 'J']).all():
    raise ValueError('Unexpected choice key; inspect events')
choices['choice_side'] = choices['key'].map({'F': 'left', 'J': 'right'})
swapped = choices['cb'].eq('y').fillna(False)
is_a = (choices['key'].eq('J') & swapped) | (choices['key'].eq('F') & ~swapped)
choices['choice_ab'] = pd.Series('B', index=choices.index)
choices.loc[is_a, 'choice_ab'] = 'A'
choices['chosen_image'] = choices['option_a'].where(is_a, choices['option_b'])
choice_columns = trial_keys + ['key', 'choice_side', 'choice_ab', 'chosen_image', 'within_trial_ms']
trials = trials.merge(choices[choice_columns].rename(columns={'key': 'choice_key', 'within_trial_ms': 'interval_before_choice_ms'}),
                      on=trial_keys, how='left', validate='one_to_one')

# Only calculate a region RT when the preceding region key was actually logged.
for region in range(1, 7):
    current, previous = f'r{region}', f'r{region - 1}'
    region_events = events.loc[events.stage.eq(current) & events.previous_stage.eq(previous), trial_keys + ['within_trial_ms']]
    if region_events.duplicated(trial_keys).any():
        raise ValueError(f'Multiple {current} reading times in a trial')
    trials = trials.merge(region_events.rename(columns={'within_trial_ms': f'{current}_ms'}),
                          on=trial_keys, how='left', validate='one_to_one')

trials = trials.sort_values(['session', 'trial_number']).reset_index(drop=True)
print(f'{trials.session.nunique()} submissions; {len(trials)} trials; {len(events)} logged keys')
print(f"Trials without a logged choice: {trials.choice_key.isna().sum()}; trials with blank cb (treated as non-y): {trials.cb.isna().sum()}")
print('Logged stages:', events.stage.value_counts().to_dict())

10 submissions; 400 trials; 3580 logged keys
Trials without a logged choice: 0; trials with blank cb (treated as non-y): 160
Logged stages: {'r0': 400, 'r1': 400, 'r5': 400, 'r2': 400, 'r3': 400, 'r4': 400, 'choice_key': 400, 'r6': 400, 'context_key': 240, 'question_key': 140}


## All events
`participant_id` is the typed ID; `prolific_pid` is the URL parameter. Rows are ordered by submission, numeric `no`, numeric `item`, and event order within each trial. Blank `no` values are preserved as missing integers and sorted last. `trial_number` records the actual presentation order; sorting does not change the already-calculated intervals. Choice and region-time columns are repeated for each event in the same trial.

In [3]:
trial_results = ['choice_key', 'choice_side', 'choice_ab', 'chosen_image',
                 'interval_before_choice_ms'] + [f'r{i}_ms' for i in range(1, 7)]
event_table = events.merge(trials[trial_keys + trial_results], on=trial_keys,
                           how='left', validate='many_to_one')
event_table = event_table.sort_values(
    ['session', 'no', 'item', 'trial_number', 'key_number_in_trial'],
    na_position='last', kind='stable').reset_index(drop=True)
session_order = sorted(trials['session'].dropna().unique())
participant_labels = {session: f'participant_{index:02d}'
                      for index, session in enumerate(session_order, start=1)}
event_table['participant'] = event_table['session'].map(participant_labels)

# Set True to exclude events from trials where exp == 2.
FILTER_EXP_2 = False
if FILTER_EXP_2:
    event_table = event_table.loc[event_table['exp'].ne(2)].copy()

display_columns = [
    'participant', 'participant_id', 'no', 'cnd', 'session', 'item', 'exp', 'pst', 'mlt', 'cb',
    'stage', 'key', 'timestamp_ms', 'previous_stage', 'key_number_in_trial',
    'within_trial_ms', 'since_previous_key_ms', 'option_a', 'option_b',
    'correct', 'choice_key', 'choice_side', 'choice_ab', 'chosen_image',
    'interval_before_choice_ms', *[f'r{i}_ms' for i in range(1, 7)],
]
event_table = event_table[display_columns]
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(event_table)

# Optional private export (do not commit this file):
# event_table.to_csv('parsed_events.csv', index=False)

,participant,participant_id,no,cnd,session,item,exp,pst,mlt,cb,stage,key,timestamp_ms,previous_stage,key_number_in_trial,within_trial_ms,since_previous_key_ms,option_a,option_b,correct,choice_key,choice_side,choice_ab,chosen_image,interval_before_choice_ms,r1_ms,r2_ms,r3_ms,r4_ms,r5_ms,r6_ms
0,participant_01,69f05ebc11ace319136e99d2,1,C1T1,1,1,1,0,1,n,context_key,SPACE,1793199506541,NaN,1,NaN,2464.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
1,participant_01,69f05ebc11ace319136e99d2,1,C1T1,1,1,1,0,1,n,r0,SPACE,1793199507277,context_key,2,736.0,736.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
2,participant_01,69f05ebc11ace319136e99d2,1,C1T1,1,1,1,0,1,n,r1,SPACE,1793199507613,r0,3,336.0,336.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
3,participant_01,69f05ebc11ace319136e99d2,1,C1T1,1,1,1,0,1,n,r2,SPACE,1793199507933,r1,4,320.0,320.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
4,participant_01,69f05ebc11ace319136e99d2,1,C1T1,1,1,1,0,1,n,r3,SPACE,1793199508253,r2,5,320.0,320.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
5,participant_01,69f05ebc11ace319136e99d2,1,C1T1,1,1,1,0,1,n,r4,SPACE,1793199508541,r3,6,288.0,288.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
6,participant_01,69f05ebc11ace319136e99d2,1,C1T1,1,1,1,0,1,n,r5,SPACE,1793199508845,r4,7,304.0,304.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
7,participant_01,69f05ebc11ace319136e99d2,1,C1T1,1,1,1,0,1,n,r6,SPACE,1793199509165,r5,8,320.0,320.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
8,participant_01,69f05ebc11ace319136e99d2,1,C1T1,1,1,1,0,1,n,choice_key,J,1793199510381,r6,9,1216.0,1216.0,1a,1b,1b,J,right,B,1b,1216.0,336.0,320.0,320.0,288.0,304.0,320.0
9,participant_01,69f05ebc11ace319136e99d2,14,C1T2,1,2,1,0,1,y,context_key,SPACE,1793199537389,NaN,1,NaN,3136.0,2a,2b,2a,J,right,A,2a,896.0,320.0,272.0,320.0,336.0,320.0,320.0


### Event-table columns
`participant` is a pseudonymous label assigned by submission order; `participant_id` is the original ID, so the table is not anonymized while that column remains. `session` identifies the submission block and is used to distinguish submissions and calculate the participant label. `within_trial_ms` is the interval since the preceding logged key in the same trial; it is missing for the first logged key in a trial. `since_previous_key_ms` is the interval since the preceding logged key in that submission, including across trial boundaries. All intervals are milliseconds.

In [4]:
# Attention checks are identified by their condition label, not by the optional event-table filter.
attention_checks = trials.loc[
    trials['cnd'].astype('string').str.startswith('attention', na=False)
].copy()
if attention_checks.empty:
    raise ValueError('No attention-check trials found')

correct_a = attention_checks['correct'].eq(attention_checks['option_a'])
correct_b = attention_checks['correct'].eq(attention_checks['option_b'])
if not (correct_a ^ correct_b).all():
    raise ValueError('Each attention-check correct answer must match exactly one of option_a or option_b')

attention_checks['correct_choice_ab'] = correct_a.map({True: 'A', False: 'B'})
attention_checks['correct_answer'] = attention_checks['correct']
swapped = attention_checks['cb'].eq('y').fillna(False)
correct_on_right = (attention_checks['correct_choice_ab'].eq('A') & swapped) | (
    attention_checks['correct_choice_ab'].eq('B') & ~swapped
)
attention_checks['correct_key'] = correct_on_right.map({True: 'J', False: 'F'})
attention_checks['passed'] = attention_checks['chosen_image'].eq(attention_checks['correct_answer']).fillna(False)
attention_checks['result'] = attention_checks['passed'].map({True: 'passed', False: 'FAILED / no response'})

attention_summary = (
    attention_checks.groupby(['session', 'participant_id', 'prolific_pid'], dropna=False)
    .agg(n_attention_checks=('item', 'size'), n_failed=('passed', lambda answers: int((~answers).sum())))
    .reset_index()
)
flagged_participants = attention_summary.loc[attention_summary['n_failed'].gt(0)].copy()
failed_checks = attention_checks.loc[~attention_checks['passed'], [
    'session', 'participant_id', 'prolific_pid', 'item', 'cnd', 'choice_key',
    'chosen_image', 'correct_choice_ab', 'correct_answer', 'correct_key', 'result'
]].sort_values(['session', 'item'], kind='stable')

print('Participants with one or more failed attention checks:')
if flagged_participants.empty:
    print('None')
else:
    display(flagged_participants)
    print('Failed attention-check responses and correct answers:')
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(failed_checks)


Participants with one or more failed attention checks:
None


## Join stimuli and visualize responses
Stimuli are joined to the event table on `no`. Events without a `no` stay in the table but have no stimulus match; condition plots use only matched stimulus trials. The plots first average within each submission, then calculate means and SD/SE across submissions.

In [5]:
STIMULI_PATH = Path('stimuli.csv')
if not STIMULI_PATH.exists():
    STIMULI_PATH = Path('new') / 'stimuli.csv'
stimuli = pd.read_csv(STIMULI_PATH)
stimuli['no'] = pd.to_numeric(stimuli['no'], errors='coerce').astype('Int64')
stimuli_with_no = stimuli.dropna(subset=['no']).copy()
if stimuli_with_no['no'].duplicated().any():
    raise ValueError('Stimuli must have at most one row per nonmissing no value')

# Keep result-log fields, while retaining colliding stimulus fields with a clear suffix.
overlapping_columns = (set(stimuli_with_no.columns) & set(event_table.columns)) - {'no'}
stimuli_for_join = stimuli_with_no.rename(
    columns={column: f'{column}_stimulus' for column in overlapping_columns}
)
event_rows_before_merge = len(event_table)
event_table = event_table.merge(
    stimuli_for_join, on='no', how='left', validate='many_to_one', indicator='stimuli_match'
)
if len(event_table) != event_rows_before_merge:
    raise ValueError('Stimuli merge changed the event-table row count')
matched_events = event_table['stimuli_match'].eq('both')
if not event_table.loc[matched_events, 'cnd'].eq(event_table.loc[matched_events, 'cnd_stimulus']).all():
    raise ValueError('Logged cnd does not match the stimulus cnd for one or more no values')

stimulus_conditions = stimuli_with_no[['no', 'context_condition', 'target_condition']]
analysis_trials = trials.merge(
    stimulus_conditions, on='no', how='left', validate='many_to_one'
)
if FILTER_EXP_2:
    analysis_trials = analysis_trials.loc[analysis_trials['exp'].ne(2)].copy()
plot_trials = analysis_trials.dropna(subset=['context_condition', 'target_condition']).copy()
plot_trials['participant'] = plot_trials['session'].map(participant_labels)
plot_trials['choice_rt_ms'] = plot_trials['interval_before_choice_ms']
if plot_trials.empty:
    raise ValueError('No matched stimulus trials available for condition plots')

print(f'Stimulus rows joined: {int(matched_events.sum()):,} events; unmatched events retained: {int((~matched_events).sum()):,}.')
print(f'Condition plots include {len(plot_trials):,} matched trials from {plot_trials.session.nunique()} submissions.')
display(event_table[['participant', 'participant_id', 'session', 'no', 'cnd',
                     'context_condition', 'target_condition', 'stimuli_match']].head())

Stimulus rows joined: 2,160 events; unmatched events retained: 1,420.
Condition plots include 240 matched trials from 10 submissions.


,participant,participant_id,session,no,cnd,context_condition,target_condition,stimuli_match
0,participant_01,69f05ebc11ace319136e99d2,1,1,C1T1,ctx-exhaustive,non-focused,both
1,participant_01,69f05ebc11ace319136e99d2,1,1,C1T1,ctx-exhaustive,non-focused,both
2,participant_01,69f05ebc11ace319136e99d2,1,1,C1T1,ctx-exhaustive,non-focused,both
3,participant_01,69f05ebc11ace319136e99d2,1,1,C1T1,ctx-exhaustive,non-focused,both
4,participant_01,69f05ebc11ace319136e99d2,1,1,C1T1,ctx-exhaustive,non-focused,both


### Reading-time and choice-interval plots
Each line shows the mean of per-submission means; the error bars default to SE across submissions. Set `ERROR_BAR = 'sd_ms'` to show SD instead. The choice measure is `interval_before_choice_ms`, the gap between the previous logged key and choice; it includes the 500 ms fixation and is not a true choice-image-onset reaction time.

In [6]:
import math
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ERROR_BAR = 'se_ms'  # Change to 'sd_ms' for standard-deviation bars.
condition_columns = ['cnd', 'context_condition', 'target_condition']
region_columns = [f'r{region}_ms' for region in range(1, 7)]
region_order = [f'r{region}' for region in range(1, 7)]

reading_long = plot_trials[
    ['session', 'participant', *condition_columns, *region_columns]
].melt(
    id_vars=['session', 'participant', *condition_columns],
    value_vars=region_columns,
    var_name='region', value_name='reading_time_ms'
).dropna(subset=['reading_time_ms'])
reading_long['region'] = reading_long['region'].str.replace('_ms', '', regex=False)

def summarize_across_submissions(data, group_columns, measure):
    per_submission = (
        data.groupby(['session', 'participant', *group_columns], dropna=False, observed=True)[measure]
        .mean().rename('submission_mean_ms').reset_index()
    )
    summary = (
        per_submission.groupby(group_columns, dropna=False, observed=True)
        .agg(mean_ms=('submission_mean_ms', 'mean'),
             sd_ms=('submission_mean_ms', 'std'),
             n_submissions=('session', 'nunique'))
        .reset_index()
    )
    summary['se_ms'] = summary['sd_ms'] / summary['n_submissions'].pow(0.5)
    return summary

if ERROR_BAR not in {'sd_ms', 'se_ms'}:
    raise ValueError("ERROR_BAR must be either 'sd_ms' or 'se_ms'")

reading_summaries = {}
for condition in condition_columns:
    summary = summarize_across_submissions(
        reading_long, [condition, 'region'], 'reading_time_ms'
    )
    reading_summaries[condition] = summary
    figure = px.line(
        summary, x='region', y='mean_ms', color=condition, markers=True,
        error_y=ERROR_BAR, category_orders={'region': region_order},
        hover_data={'sd_ms': ':.1f', 'se_ms': ':.1f', 'n_submissions': True},
        labels={'region': 'Sentence region', 'mean_ms': 'Mean reading time (ms)',
                condition: condition.replace('_', ' ').title()},
        title=f'Sentence-region reading times by {condition.replace("_", " ")} (±{ERROR_BAR[:2].upper()})'
    )
    figure.update_layout(hovermode='x unified')
    figure.show()

choice_summaries = {}
for condition in condition_columns:
    summary = summarize_across_submissions(
        plot_trials.dropna(subset=['choice_rt_ms']), [condition], 'choice_rt_ms'
    ).sort_values(condition, kind='stable')
    choice_summaries[condition] = summary
    figure = px.bar(
        summary, x=condition, y='mean_ms', error_y=ERROR_BAR,
        hover_data={'sd_ms': ':.1f', 'se_ms': ':.1f', 'n_submissions': True},
        labels={condition: condition.replace('_', ' ').title(),
                'mean_ms': 'Mean logged choice interval (ms)'},
        title=f'Logged choice intervals by {condition.replace("_", " ")} (±{ERROR_BAR[:2].upper()})'
    )
    figure.update_layout(hovermode='x unified')
    figure.show()

print('Reading-time and choice-interval summaries are available in reading_summaries and choice_summaries.')

Reading-time and choice-interval summaries are available in reading_summaries and choice_summaries.


In [7]:
choice_data = plot_trials.dropna(subset=['choice_ab']).copy()
if choice_data.empty:
    raise ValueError('No mapped A/B choices available for donut plots')

def choice_donut_figure(data, condition):
    levels = sorted(data[condition].dropna().unique(), key=str)
    columns = min(3, len(levels))
    rows = math.ceil(len(levels) / columns)
    specs = [
        [{'type': 'domain'} if column < len(levels) - row * columns else None
         for column in range(columns)]
        for row in range(rows)
    ]
    figure = make_subplots(
        rows=rows, cols=columns, specs=specs,
        subplot_titles=[str(level) for level in levels]
    )
    for index, level in enumerate(levels):
        row, column = divmod(index, columns)
        counts = (
            data.loc[data[condition].eq(level), 'choice_ab']
            .value_counts().reindex(['A', 'B'], fill_value=0)
        )
        figure.add_trace(
            go.Pie(
                labels=counts.index, values=counts.values, hole=0.5, sort=False,
                textinfo='label+percent', showlegend=index == 0,
                marker={'colors': ['#4C78A8', '#F58518']},
                hovertemplate='<b>Choice %{label}</b><br>Trials: %{value}<br>Share: %{percent}<extra></extra>'
            ), row=row + 1, col=column + 1
        )
    figure.update_layout(
        title=f'A/B choices by {condition.replace("_", " ")}',
        height=max(360, rows * 320), width=max(700, columns * 340),
        legend_title_text='Choice', margin={'t': 70, 'b': 20}
    )
    return figure

choice_donut_figures = {}
for condition in condition_columns:
    choice_donut_figures[condition] = choice_donut_figure(choice_data, condition)
    choice_donut_figures[condition].show()